# Experiment 2 – Identity-Term Swap

Goal: measure whether identity words alone push harmless sentences toward 'toxic' in Detoxify (original vs unbiased), how much of the decision SHAP places on the identity word, and how stable LIME's explanations are across random seeds. Runtime: Colab → Runtime → Change runtime type → T4 GPU.

## Cell 1: install, imports and shared helpers

In [ ]:
!pip -q install detoxify shap lime
import os, re, numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, torch, shap, lime
from detoxify import Detoxify
from lime.lime_text import LimeTextExplainer
from scipy.stats import spearmanr
from importlib.metadata import version   # lime has no __version__ attribute
print("shap", shap.__version__, "| lime", version("lime"), "| torch", torch.__version__)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
OUT = "outputs_exp2"; os.makedirs(OUT, exist_ok=True)
BG, FG = "#FFFFFF", "#000000"                     # pink theme: white background, black text, pink bars
plt.rcParams.update({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14,
                     "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
                     "axes.edgecolor": FG, "axes.labelcolor": FG, "text.color": FG,
                     "xtick.color": FG, "ytick.color": FG, "legend.labelcolor": FG})
COLORS = {"original": "#D6006F", "unbiased": "#FF8FC4", "control": "#9A8590",
          "n500": "#A3134F", "n5000": "#FF7AB8", "err": "#333333"}   # side-by-side pairs differ strongly in lightness          # side-by-side pairs differ strongly in lightness
PINK_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list("pink_black", ["#FFF0F6", "#FF4FA3", "#7A0A3E"])

MODELS = {m: Detoxify(m, device=DEVICE) for m in ["original", "unbiased"]}

def proba(texts, model="original", bs=64):
    texts = [str(t) for t in texts]
    out = []
    for i in range(0, len(texts), bs):
        out.extend(np.atleast_1d(MODELS[model].predict(texts[i:i+bs])["toxicity"]))
    return np.array(out, dtype=float)

def logit(texts, model="original"):
    p = np.clip(proba(texts, model), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))                       # log-odds: additive SHAP (shap-exercises)

def lime_fn(model="original"):
    def f(texts):
        p = proba(texts, model)
        return np.column_stack([1 - p, p])           # friday-03 pattern
    return f

MASKER = shap.maskers.Text(r"\s+")   # split on spaces: r"\W+" drops the final full stop, which breaks additivity
def shap_explainer(model="original"):
    return shap.Explainer(lambda t: logit(t, model), MASKER)   # single output: toxicity log-odds

def check_efficiency(sv, texts, model, tol=1e-2):
    f = logit(texts, model)
    recon = np.array([float(sv[i].base_values) + sv[i].values.sum() for i in range(len(texts))])
    err = np.abs(recon - f).max()
    assert err < tol, f"Efficiency check FAILED, max error {err}"
    print(f"[{model}] efficiency check passed, max error = {err:.2e}")

def shap_word_weights(e):
    w = {}
    for tok, v in zip(e.data, e.values):
        k = re.sub(r"\W+", "", tok).lower()
        if k: w[k] = w.get(k, 0.0) + float(v)
    return w

def lime_word_weights(text, model="original", seed=SEED, num_samples=2000):
    expl = LimeTextExplainer(class_names=["non-toxic", "toxic"], random_state=seed)
    n = len(set(re.findall(r"\w+", text)))
    exp = expl.explain_instance(text, lime_fn(model), num_features=n,
                                num_samples=num_samples, labels=(1,))
    return {w.lower(): v for w, v in exp.as_list(label=1)}, exp

def agreement(sw, lw, k=3):
    common = sorted(set(sw) & set(lw))
    rho = spearmanr([sw[w] for w in common], [lw[w] for w in common]).correlation if len(common) > 2 else np.nan
    top = lambda d, n: [w for w in sorted(d, key=lambda w: -d[w])[:n]]
    return dict(rho=rho, top1_agree=top(sw, 1) == top(lw, 1),
                top3_overlap=len(set(top(sw, k)) & set(top(lw, k))) / k)

## Cell 2: build the template grid (do not edit)

In [ ]:
TEMPLATES = ["I am a {} person.", "My neighbour is {} and she is lovely.",
             "{} people are welcome here.", "My best friend is {}.",
             "I grew up in a {} family.", "Being {} is completely normal.",
             "We hired a {} engineer last week.", "She is a proud {} woman.",
             "He is {} and works as a teacher.", "Our new {} classmate joined the team."]
TERMS = ["gay","lesbian","transgender","black","white","asian","muslim","jewish",
         "christian","immigrant","deaf","blind","disabled","poor","rural"]
CONTROLS = ["tall","young","local","quiet"]
grid = pd.DataFrame([(t, w, w in CONTROLS, t.format(w)) for t in TEMPLATES for w in TERMS + CONTROLS],
                    columns=["template","term","is_control","text"])
print(len(grid), "sentences")   # expect 190
grid.to_csv(f"{OUT}/exp2_sentences.csv", index=False)

## Cell 3: scores and the gap from the control words

`gap` = the sentence's score minus the mean score of the 4 control words in the **same template**.

In [ ]:
for m in MODELS:
    grid[f"p_{m}"] = proba(grid.text, m)
    ctrl = grid[grid.is_control].groupby("template")[f"p_{m}"].mean()
    grid[f"gap_{m}"] = grid[f"p_{m}"] - grid.template.map(ctrl)
print(grid.groupby("term")[["p_original","p_unbiased","gap_original","gap_unbiased"]].mean()
          .sort_values("gap_original", ascending=False).round(3))

## Cell 4: SHAP on the identity word (the headline XAI measure)

For each sentence, record:
- `phi_term`: the signed SHAP value of the identity or control word.
- `phi_share`: that word's share of the sentence's total |SHAP|.

In [ ]:
SV = {}
for m in MODELS:
    SV[m] = shap_explainer(m)(list(grid.text), max_evals=300, batch_size=64)
    check_efficiency(SV[m], list(grid.text), m)
    ww = [shap_word_weights(SV[m][i]) for i in range(len(grid))]
    grid[f"phi_term_{m}"] = [ww[i].get(grid.term[i], np.nan) for i in range(len(grid))]
    grid[f"phi_share_{m}"] = [abs(grid[f"phi_term_{m}"][i]) / max(np.abs(SV[m][i].values).sum(), 1e-9)
                              for i in range(len(grid))]
grid.to_csv(f"{OUT}/exp2_results.csv", index=False)

## Cell 5: Figure 1, SHAP on the identity word (slide 7 main chart)

Horizontal grouped bars: one row per term, two bars (original, unbiased) = mean signed SHAP value, error bars = std across the 10 templates. Sorted by the original model's value, largest at the top. Control-word labels greyed out. Vertical line at 0. Follows the lecturer's `barh` + `xerr` style (simple-permutation notebook).

In [ ]:
s = grid.groupby("term")[["phi_term_original","phi_term_unbiased"]].agg(["mean","std"])
s = s.sort_values(("phi_term_original","mean"), ascending=True)
y = np.arange(len(s)); h = 0.4
fig, ax = plt.subplots(figsize=(9, 8))
for j, m in enumerate(["original","unbiased"]):
    ax.barh(y + (j-0.5)*h, s[(f"phi_term_{m}","mean")], h, xerr=s[(f"phi_term_{m}","std")],
            capsize=2, ecolor=COLORS["err"], error_kw=dict(elinewidth=1.5), color=COLORS[m], label=f"Detoxify {m}")
ax.set_yticks(y); ax.set_yticklabels(s.index)
for lbl in ax.get_yticklabels():
    if lbl.get_text() in CONTROLS: lbl.set_color(COLORS["control"])
ax.axvline(0, color=FG, lw=0.8)
ax.set_xlabel("SHAP value of the group word\n(→ pushes toward 'toxic', log-odds)")
ax.set_title("Does the group word alone push toward removal?")
ax.legend(frameon=False, loc="lower right"); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_exp2_identity_shap.png", dpi=200)

## Cell 6: Figure 2, score heatmap (slide 7 small inset)

Rows are the terms (same order as Figure 1, top = largest), columns are original and unbiased, and cells show mean P(toxic) with the value written in each cell. Sequential light-to-deep pink colormap (`PINK_CMAP`, pink theme in place of the spec's `Blues`).

In [ ]:
hm = grid.groupby("term")[["p_original","p_unbiased"]].mean().loc[s.index[::-1]]
fig, ax = plt.subplots(figsize=(4.5, 8))
im = ax.imshow(hm.values, cmap=PINK_CMAP, aspect="auto", vmin=0, vmax=max(hm.values.max(), 0.05))
ax.set_xticks([0,1]); ax.set_xticklabels(["original","unbiased"]); ax.set_yticks(range(len(hm)))
ax.set_yticklabels(hm.index)
for r in range(hm.shape[0]):
    for c in range(2):
        v = hm.values[r, c]
        ax.text(c, r, f"{v:.3f}", ha="center", va="center", fontsize=11,
                color="white" if v > 0.6 * hm.values.max() else "black")
ax.set_title("Mean P(toxic)\n(harmless sentences)")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_exp2_score_heatmap.png", dpi=200)

## Cell 7: did the debiasing remove the reliance, or just lower the score?

Per term (identity terms only), the mean `gap` and mean `phi_share` for each model.

In [ ]:
ids = grid[~grid.is_control]
t = ids.groupby("term")[["gap_original","gap_unbiased","phi_share_original","phi_share_unbiased"]].mean()
t["gap_reduction_%"] = 100 * (1 - t.gap_unbiased / t.gap_original.replace(0, np.nan))
print(t.sort_values("gap_original", ascending=False).round(3))
t.to_csv(f"{OUT}/exp2_debias_comparison.csv")

> **Interpretation hint:** *If `gap_unbiased` is near 0 but `phi_share_unbiased` is still high, the debiased model lowered the score but the decision still leans on the group word.*

## Cell 8: LIME stability test (slide 8 left chart)

The **5 identity-term sentences with the highest `phi_term_original`**. LIME is run 10 times each (seeds 0–9) at two sample sizes, 500 and 5000 (5000 is LIME's default, W5-L02 p2). For every run we record the identity word's weight and rank, and the top word. SHAP is also re-run twice on the same 5 sentences to show it is deterministic.

In [ ]:
picks = ids.sort_values("phi_term_original", ascending=False).head(5)[["text","term"]].values.tolist()
stab = []
for text, term in picks:
    for ns in [500, 5000]:
        for seed in range(10):
            lw, _ = lime_word_weights(text, "original", seed=seed, num_samples=ns)
            ranked = sorted(lw, key=lambda w: -lw[w])
            stab.append(dict(text=text, term=term, n=ns, seed=seed, weight=lw.get(term, np.nan),
                             rank=ranked.index(term) + 1 if term in ranked else np.nan, top_word=ranked[0],
                             **{f"w_{w}": v for w, v in lw.items()}))
stab = pd.DataFrame(stab); stab.to_csv(f"{OUT}/exp2_lime_stability.csv", index=False)
# grouped per sentence (not per term): the 5 picks can share the same identity word
summary = stab.groupby(["text","term","n"]).agg(weight_mean=("weight","mean"), weight_std=("weight","std"),
                                                 distinct_top_words=("top_word","nunique"),
                                                 rank_min=("rank","min"), rank_max=("rank","max"))
print(summary.round(4))

sv_a = shap_explainer("original")([p[0] for p in picks], max_evals=300)
sv_b = shap_explainer("original")([p[0] for p in picks], max_evals=300)
shap_diff = max(np.abs(sv_a[i].values - sv_b[i].values).max() for i in range(len(picks)))
print(f"SHAP max difference between two runs: {shap_diff:.2e}  (≈0 means deterministic)")

## Cell 9: Figure 3, LIME stability chart (slide 8)

The **first** pick sentence. One group of bars per word in the sentence (sentence order), two bars per word (n=500 and n=5000), bar = mean LIME weight over the 10 seeds, error bar = std. The identity word's x-label is bold.

In [ ]:
text0, term0 = picks[0]
words = [w.lower() for w in re.findall(r"\w+", text0)]
sub = stab[stab.text == text0]
x = np.arange(len(words)); fig, ax = plt.subplots(figsize=(10, 5))
for j, ns in enumerate([500, 5000]):
    d = sub[sub.n == ns]
    means = [d[f"w_{w}"].mean() for w in words]; stds = [d[f"w_{w}"].std() for w in words]
    ax.bar(x + (j-0.5)*0.38, means, 0.38, yerr=stds, capsize=4, ecolor=COLORS["err"], error_kw=dict(elinewidth=1.5),
           color=COLORS[f"n{ns}"], label=f"{ns} samples per run")
ax.set_xticks(x); ax.set_xticklabels(words)
for lbl in ax.get_xticklabels():
    if lbl.get_text() == term0: lbl.set_fontweight("bold")
ax.axhline(0, color=FG, lw=0.8); ax.set_ylabel("LIME weight toward 'toxic'")
ax.set_title("Same sentence, 10 LIME runs: how much does the answer move?")
ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_exp2_lime_stability.png", dpi=200)

## Cell 10: SHAP–LIME agreement on 40 sentences

40 identity-term rows sampled with `random_state=SEED`; `agreement()` between SHAP (original) and a single LIME run (seed 42, 2000 samples).

In [ ]:
samp = ids.sample(40, random_state=SEED)
ag = []
for i in samp.index:
    lw, _ = lime_word_weights(grid.text[i], "original")
    a = agreement(shap_word_weights(SV["original"][i]), lw)
    a["identity_top_in_both"] = (max(lw, key=lw.get) == grid.term[i]) and \
        (max(shap_word_weights(SV["original"][i]).items(), key=lambda kv: kv[1])[0] == grid.term[i])
    ag.append(a)
ag = pd.DataFrame(ag, index=samp.index); ag.to_csv(f"{OUT}/exp2_agreement.csv")

## Cell 10b (extension, not in the original spec): LIME stability on edge cases

Cell 8 only tested the 5 sentences with the **strongest** SHAP signal, all "gay" sentences where one word dominates. That is LIME's easiest case. Here the same 10-seed × {500, 5000} test is run on harder cases, each **picked by a fixed rule** (no hand-picking):

| Case | Rule | Why it is hard |
|---|---|---|
| strong (reference) | highest `phi_term_original` (same as Cell 8's first pick) | baseline |
| weak signal | `phi_term_original` closest to 0 | small effect, so noise matters more |
| near-tie top words | smallest SHAP gap between the #1 and #2 words | most likely to flip the top word |
| negative push | most negative `phi_term_original` | group word pushes *away* from toxic |
| SHAP≠LIME | Cell 10 top-word disagreement with the lowest rho | methods already disagree here |
| control word | control word with the largest \|SHAP\| | LIME should say it barely matters |
| strong, unbiased model | the strong sentence on Detoxify `unbiased` | same text, weaker signal |

Each sentence is used once (if a rule picks a sentence already taken, it moves to the next candidate). Also records LIME's own fit quality (R² of its local surrogate). Runtime: about 1.5× Cell 8.

**Fresh Colab session?** Run Cell 1, then upload `exp2_results.csv` and `exp2_agreement.csv` into the `outputs_exp2/` folder before running this cell. It reloads them and recomputes the SHAP values it needs (a few minutes).

In [ ]:
if "grid" not in globals():                                   # fresh session: reload saved results
    grid = pd.read_csv(f"{OUT}/exp2_results.csv")
if "ag" not in globals():
    ag = pd.read_csv(f"{OUT}/exp2_agreement.csv", index_col=0)
if "SV" not in globals():
    SV = {"original": shap_explainer("original")(list(grid.text), max_evals=300, batch_size=64)}
ids = grid[~grid.is_control]

def shap_margin(i):
    v = sorted(shap_word_weights(SV["original"][i]).values(), reverse=True)
    return v[0] - v[1] if len(v) > 1 else np.inf

strong_i = ids.phi_term_original.idxmax()
cands = {
    "strong (reference)": [strong_i],
    "weak signal": list(ids.phi_term_original.abs().sort_values().index),
    "near-tie top words": sorted(ids.index, key=shap_margin),
    "negative push": list(ids.phi_term_original.sort_values().index),
    "SHAP≠LIME": list(ag[~ag.top1_agree].sort_values("rho").index) + list(ag.sort_values("rho").index),
    "control word": list(grid[grid.is_control].phi_term_original.abs().sort_values(ascending=False).index),
}
EDGE, used = [], set()
for case, idx in cands.items():
    i = next(j for j in idx if j not in used); used.add(i)
    EDGE.append((case, i, "original"))
EDGE.append(("strong, unbiased model", strong_i, "unbiased"))
for case, i, m in EDGE:
    print(f"{case:24s} [{m}] {grid.text[i]!r:45s} phi_term_original = {grid.phi_term_original[i]:+.3f}")

edge = []
for case, i, m in EDGE:
    text, term = grid.text[i], grid.term[i]
    for ns in [500, 5000]:
        for seed in range(10):
            lw, exp = lime_word_weights(text, m, seed=seed, num_samples=ns)
            ranked = sorted(lw, key=lambda w: -lw[w])
            edge.append(dict(case=case, model=m, text=text, term=term, n=ns, seed=seed,
                             weight=lw.get(term, np.nan), rank=ranked.index(term) + 1 if term in ranked else np.nan,
                             top_word=ranked[0], top3=" ".join(ranked[:3]), r2=exp.score))
edge = pd.DataFrame(edge); edge.to_csv(f"{OUT}/exp2_lime_edge_cases.csv", index=False)

es = edge.groupby(["case","n"], sort=False).agg(
    text=("text","first"), term=("term","first"), weight_mean=("weight","mean"), weight_std=("weight","std"),
    sign_flip=("weight", lambda w: bool((w > 0).any() and (w < 0).any())),
    distinct_top_words=("top_word","nunique"), distinct_top3=("top3","nunique"),
    rank_min=("rank","min"), rank_max=("rank","max"), r2_mean=("r2","mean"))
es["rel_spread_%"] = 100 * es.weight_std / es.weight_mean.abs().replace(0, np.nan)
es.to_csv(f"{OUT}/exp2_lime_edge_summary.csv")
print(es.drop(columns="text").round(4).to_string())

print("\n=== EDGE-CASE SUMMARY ===")
for case, _, m in EDGE:
    a, b = es.loc[(case, 500)], es.loc[(case, 5000)]
    print(f"{case:24s} '{a.term}' | top word changed: {'YES' if a.distinct_top_words > 1 else 'no'} @500, "
          f"{'YES' if b.distinct_top_words > 1 else 'no'} @5000 | group-word rank {a.rank_min:.0f}-{a.rank_max:.0f} @500 | "
          f"spread {a['rel_spread_%']:.1f}% @500, {b['rel_spread_%']:.1f}% @5000 | sign flip @500: {a.sign_flip} | R² {a.r2_mean:.2f}")
changed500 = sum(es.loc[(c, 500), "distinct_top_words"] > 1 for c, _, _ in EDGE)
changed5000 = sum(es.loc[(c, 5000), "distinct_top_words"] > 1 for c, _, _ in EDGE)
print(f"\nTop word changed across 10 seeds: {changed500}/{len(EDGE)} edge cases at n=500, {changed5000}/{len(EDGE)} at n=5000")

## Cell 10c (extension): Figure 4, LIME stability across edge cases (slide 8 backup or replacement)

One row per edge case (top = strongest signal), two bars = the run-to-run spread of the group word's LIME weight relative to its size (std ÷ |mean| over 10 seeds), log scale. The text on the right says whether the top word changed across seeds.

In [ ]:
cases = [c for c, _, _ in EDGE][::-1]
y = np.arange(len(cases)); h = 0.4
fig, ax = plt.subplots(figsize=(10, 6.5))
for j, ns in enumerate([500, 5000]):
    v = [max(es.loc[(c, ns), "rel_spread_%"], 1e-3) for c in cases]
    ax.barh(y + (j-0.5)*h, v, h, color=COLORS[f"n{ns}"], label=f"{ns} samples per run")
ax.set_xscale("log")
ax.set_yticks(y); ax.set_yticklabels([f"{c}\n'{es.loc[(c, 500), 'term']}'" for c in cases], fontsize=12)
for k, c in enumerate(cases):
    d500, d5000 = es.loc[(c, 500), "distinct_top_words"], es.loc[(c, 5000), "distinct_top_words"]
    txt = "top word stable" if max(d500, d5000) == 1 else f"top word changed\n(distinct top words: {d500} @500, {d5000} @5000)"
    ax.text(1.02, k, txt, transform=ax.get_yaxis_transform(), va="center", fontsize=11,
            color=FG if max(d500, d5000) == 1 else COLORS["n500"])
ax.set_xlabel("Run-to-run spread of the group word's LIME weight\n(std ÷ |mean| over 10 seeds, %, log scale)")
ax.set_title("Is LIME still stable when the answer isn't obvious?")
ax.legend(frameon=False, loc="lower right"); ax.spines[["top","right"]].set_visible(False)
fig.savefig(f"{OUT}/fig_exp2_lime_edge_cases.png", dpi=200, bbox_inches="tight")

## Cell 11: results summary (numbers to quote on the slides)

In [ ]:
print("=== EXPERIMENT 2 SUMMARY ===")
top = t.sort_values("gap_original", ascending=False).head(5)
print("Top 5 terms by score gap (original):"); print(top[["gap_original","gap_unbiased"]].round(3))
print(f"Mean gap, identity terms: original {ids.gap_original.mean():.3f} | unbiased {ids.gap_unbiased.mean():.3f}")
print(f"Mean share of |SHAP| on the identity word: original {ids.phi_share_original.mean():.0%} "
      f"| unbiased {ids.phi_share_unbiased.mean():.0%}")
flips500 = (summary.xs(500, level="n").distinct_top_words > 1).sum()
flips5000 = (summary.xs(5000, level="n").distinct_top_words > 1).sum()
print(f"LIME top word changed across 10 seeds: {flips500}/5 sentences at n=500, {flips5000}/5 at n=5000")
print(f"Mean std of identity-word LIME weight: n=500 {summary.xs(500, level='n').weight_std.mean():.4f} "
      f"| n=5000 {summary.xs(5000, level='n').weight_std.mean():.4f}")
print(f"SHAP run-to-run max difference: {shap_diff:.2e}")
print(f"SHAP and LIME agreed on the TOP word in {int(ag.top1_agree.sum())}/40 sentences; "
      f"median rho {np.nanmedian(ag.rho):.2f}; both ranked the identity word top in {int(ag.identity_top_in_both.sum())}/40")

## Cell 12: download everything

In [ ]:
!zip -qr outputs_exp2.zip outputs_exp2
from google.colab import files; files.download("outputs_exp2.zip")